Here we will try to make use of the extracted type 2,4,5 data from the rsp_events file.  

In [3]:
import json
import numpy as np

# Load the JSON file
with open("solar_events_cleaned.json", "r") as f:
    data = json.load(f)

# Lists to hold each burst type
type_I = []
type_II = []
type_III = []  
type_IV = []
type_V = []

# Loop through each event and classify by burst type in "particulars"
for event in data:
    part = event.get("particulars", "")
    if part.startswith("I/"):
        type_I.append(event)
    elif part.startswith("II/"):
        type_II.append(event)
    elif part.startswith("III/"):
        type_III.append(event)
    elif part.startswith("IV/"):
        type_IV.append(event)
    elif part.startswith("V/"):
        type_V.append(event)

# Convert to numpy arrays for shape
type_I = np.array(type_I)
type_II = np.array(type_II)
type_III = np.array(type_III)
type_IV = np.array(type_IV)
type_V = np.array(type_V)

# Print the shapes of each burst type
print("Type I shape:", type_I.shape)
print("Type II shape:", type_II.shape)
print("Type III shape:", type_III.shape)
print("Type IV shape:", type_IV.shape)
print("Type V shape:", type_V.shape)

Type I shape: (0,)
Type II shape: (245,)
Type III shape: (5348,)
Type IV shape: (138,)
Type V shape: (116,)


Since most of the data in the offline dataset is not labelled. We will now try to find the find and label the data indexes which are in the offline data from the help of the rsp file. We will save the indexes of the offline data file which does not have corresponding file in the rsp. We will append the h5 file in labels keys with the type from rsp. To append the labels key the condition that the data should not have corresponding label in the h5 file should be met. If the h5 file has label for an image and also for the rsp file, the label would not be replaced. The file will checked via date and time from rsp and timestamp key from the h5 file. The indexes of timestamps for which the labels were appended will also be saved in another variable. But the not found timestamps indexes will be saved in a seperate json file. 

In [5]:
import json

# Load the JSON file
with open("solar_events_cleaned.json", "r") as f:
    data = json.load(f)

date_begin_to_part = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("particulars")
    if date and begin and part and begin != "////":
        # Format begin as HH:MM:00.000000
        hour = begin[:2]
        minute = begin[2:]
        formatted = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        date_begin_to_part[formatted] = part_clean

# Example: print first 10 entries
for k, v in list(date_begin_to_part.items())[:10]:
    print(f"{k}: {v}")

print("Length of dictionary:", len(date_begin_to_part))

2022-05-02_00:00:00.000000: III
2022-05-02_02:49:00.000000: III
2022-05-02_03:06:00.000000: III
2022-05-02_03:42:00.000000: III
2022-05-02_04:15:00.000000: III
2022-05-02_05:15:00.000000: III
2022-05-02_06:49:00.000000: III
2022-05-02_09:29:00.000000: III
2022-05-02_10:20:00.000000: III
2022-05-02_19:08:00.000000: III
Length of dictionary: 5758


In [7]:
import json

with open("solar_events_cleaned.json", "r") as f:
    data = json.load(f)

key_counts = {}
for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("particulars")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        key_counts[key] = key_counts.get(key, 0) + 1

duplicates = {k: v for k, v in key_counts.items() if v > 1}
print(f"Number of duplicate keys: {len(duplicates)}")
for k, v in duplicates.items():
    print(f"{k}: {v} entries")

Number of duplicate keys: 88
2022-05-05_00:00:00.000000: 2 entries
2022-05-12_12:53:00.000000: 2 entries
2022-05-25_17:32:00.000000: 2 entries
2022-06-14_15:44:00.000000: 2 entries
2022-07-05_04:00:00.000000: 2 entries
2022-07-15_10:05:00.000000: 2 entries
2022-08-15_15:53:00.000000: 2 entries
2022-08-18_12:01:00.000000: 2 entries
2022-08-25_11:24:00.000000: 2 entries
2022-08-27_02:12:00.000000: 2 entries
2022-08-27_10:27:00.000000: 2 entries
2022-08-27_16:23:00.000000: 2 entries
2022-09-18_02:02:00.000000: 2 entries
2022-09-23_13:50:00.000000: 2 entries
2022-11-12_18:02:00.000000: 2 entries
2022-12-14_03:58:00.000000: 2 entries
2022-12-19_08:17:00.000000: 2 entries
2023-01-26_02:47:00.000000: 2 entries
2023-02-10_01:48:00.000000: 2 entries
2023-03-03_17:53:00.000000: 2 entries
2023-03-05_01:17:00.000000: 2 entries
2023-03-17_06:46:00.000000: 2 entries
2023-03-18_07:11:00.000000: 2 entries
2023-03-18_07:22:00.000000: 2 entries
2023-03-30_07:34:00.000000: 2 entries
2023-04-30_02:11:00.0

The reason you have 714 entries in your JSON file but only 697 unique keys in your dictionary is because some entries have the same combination of date and begin (i.e., the same timestamp key). When you use a dictionary, if two or more events have the same date and begin, the last one will overwrite the previous ones.

Some of the duplicates have same same types but some have different types of burst at the same time instant. Eg, 2013-05-31_19:57:00.000000: 2 entries

Merging part (bursts) of duplicate keys with a "/". These images have 2 bursts for 1.

In [9]:
import json

# Load the JSON file
with open("solar_events_cleaned.json", "r") as f:
    data = json.load(f)

roman_to_int = {"I": "1", "II": "2", "III": "3", "IV": "4", "V": "5"}

date_begin_to_parts = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("particulars")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        # Replace Roman numeral with integer
        for roman, integer in roman_to_int.items():
            if part_clean == roman:
                part_clean = integer
        if key in date_begin_to_parts:
            existing = date_begin_to_parts[key].split('/')
            if part_clean not in existing:
                date_begin_to_parts[key] += f"/{part_clean}"
        else:
            date_begin_to_parts[key] = part_clean

print("json dict:",(date_begin_to_parts))

json dict: {'2022-05-02_00:00:00.000000': '3', '2022-05-02_02:49:00.000000': '3', '2022-05-02_03:06:00.000000': '3', '2022-05-02_03:42:00.000000': '3', '2022-05-02_04:15:00.000000': '3', '2022-05-02_05:15:00.000000': '3', '2022-05-02_06:49:00.000000': '3', '2022-05-02_09:29:00.000000': '3', '2022-05-02_10:20:00.000000': '3', '2022-05-02_19:08:00.000000': '3', '2022-05-03_00:14:00.000000': '3', '2022-05-03_01:33:00.000000': '3', '2022-05-03_04:32:00.000000': '3', '2022-05-03_05:38:00.000000': '3', '2022-05-03_07:04:00.000000': '3', '2022-05-03_07:58:00.000000': '3', '2022-05-03_09:26:00.000000': '3', '2022-05-03_10:01:00.000000': '3', '2022-05-03_11:26:00.000000': '3', '2022-05-03_12:37:00.000000': '3', '2022-05-03_13:40:00.000000': '3', '2022-05-03_13:57:00.000000': '3', '2022-05-03_15:46:00.000000': '3', '2022-05-03_19:58:00.000000': '3', '2022-05-03_20:05:00.000000': '4', '2022-05-03_20:42:00.000000': '4', '2022-05-04_00:00:00.000000': '4', '2022-05-04_00:28:00.000000': '3', '2022-05

In above I was having problem with finding overlapping keys because, the timestamps in offile dataset has timestamps minutes as intervals of 15 (eg, 10:00, 10:15, 10:30), and the rsp has exact moments of the bursts which is not divided into intervals of 15 minutes

Make a variable of the timestamps which exists in both the data.

In [10]:
import h5py
import re
import json
from datetime import datetime, timedelta

# Function to approximate minutes to nearest 15-min interval
def approximate_minutes(timestamp):
    # Extract date and time parts
    parts = timestamp.split('_')
    date_part = parts[0]
    time_parts = parts[1].split(':')
    hour = int(time_parts[0])
    minute = int(time_parts[1])
    
    # Round to nearest 15 minutes
    if minute < 7.5:
        new_minute = 0
        new_hour = hour
    elif minute < 22.5:
        new_minute = 15
        new_hour = hour
    elif minute < 37.5:
        new_minute = 30
        new_hour = hour
    elif minute < 52.5:
        new_minute = 45
        new_hour = hour
    else:
        new_minute = 0
        new_hour = (hour + 1) % 24  # Handle hour rollover
        if new_hour == 0 and hour == 23:
            # This would be a date rollover - we'll ignore this edge case for now
            pass
    
    # Format the new timestamp
    return f"{date_part}_{new_hour:02d}:{new_minute:02d}:00.000000"

# Load the dictionary with burst types
with open("solar_events_cleaned.json", "r") as f:
    data = json.load(f)

# Create the dictionary with approximated times
roman_to_int = {"I": "1", "II": "2", "III": "3", "IV": "4", "V": "5"}
date_begin_to_parts = {}
approx_to_original = {}  # To keep track of mapping

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("particulars")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        original_key = f"{date}_{hour}:{minute}:00.000000"
        # Approximate the time
        approx_key = approximate_minutes(original_key)
        # Store mapping from approximated to original
        approx_to_original[approx_key] = original_key
        
        part_clean = part.split('/')[0]
        for roman, integer in roman_to_int.items():
            if part_clean == roman:
                part_clean = integer
                
        if approx_key in date_begin_to_parts:
            existing = date_begin_to_parts[approx_key].split('/')
            if part_clean not in existing:
                date_begin_to_parts[approx_key] += f"/{part_clean}"
        else:
            date_begin_to_parts[approx_key] = part_clean

# Open H5 file and process timestamps with approximation
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]
    
    # Extract date-time and create approximation mapping
    dt_pattern = re.compile(r'\d{4}-\d{2}-\d{2}_\d{2}:\d{2}:\d{2}\.\d+')
    dt_to_index = {}
    h5_approx_to_original = {}
    
    for i, ts in enumerate(timestamps):
        decoded = ts.decode()
        match = dt_pattern.search(decoded)
        if match:
            original_dt = match.group(0)
            approx_dt = approximate_minutes(original_dt)
            dt_to_index[approx_dt] = i
            h5_approx_to_original[approx_dt] = original_dt
    
    # Find overlapping timestamps using approximated keys
    overlapping_keys = []
    overlapping_indices = []
    overlapping_original_keys = []
    
    for approx_key in date_begin_to_parts:
        if approx_key in dt_to_index:
            overlapping_keys.append(approx_key)
            overlapping_indices.append(dt_to_index[approx_key])
            overlapping_original_keys.append((approx_to_original[approx_key], h5_approx_to_original[approx_key]))
    
    # Find non-overlapping keys
    non_overlapping_keys = [key for key in date_begin_to_parts if key not in dt_to_index]

# Print results
print(f"\nFound {len(overlapping_keys)} timestamps that exist in both sources after approximation")
print(f"First 5 overlapping approximated timestamps: {overlapping_keys[:5]}")
print(f"First 5 corresponding H5 indices: {overlapping_indices[:5]}")

if overlapping_original_keys:
    print("\nSample original timestamp pairs (RSP, H5):")
    for i, (rsp_orig, h5_orig) in enumerate(overlapping_original_keys[:5]):
        print(f"  {i+1}. RSP: {rsp_orig} -> H5: {h5_orig}")

print(f"\nFound {len(non_overlapping_keys)} timestamps in RSP that don't exist in H5")


Found 469 timestamps that exist in both sources after approximation
First 5 overlapping approximated timestamps: ['2022-05-02_06:45:00.000000', '2022-05-02_09:30:00.000000', '2022-05-02_10:15:00.000000', '2022-05-03_07:00:00.000000', '2022-05-03_08:00:00.000000']
First 5 corresponding H5 indices: [73, 65, 83, 286, 257]

Sample original timestamp pairs (RSP, H5):
  1. RSP: 2022-05-02_06:49:00.000000 -> H5: 2022-05-02_06:45:00.000000
  2. RSP: 2022-05-02_09:29:00.000000 -> H5: 2022-05-02_09:30:00.000000
  3. RSP: 2022-05-02_10:20:00.000000 -> H5: 2022-05-02_10:15:00.000000
  4. RSP: 2022-05-03_07:04:00.000000 -> H5: 2022-05-03_07:00:00.000000
  5. RSP: 2022-05-03_07:58:00.000000 -> H5: 2022-05-03_08:00:00.000000

Found 5079 timestamps in RSP that don't exist in H5


We only got 469 overlapping timestamps which suggests that the LOFAR data is indeed has european observation and solarmonitor has american observation. So since we have 5079 type 2,3,4,5 we can create artifical dynamic spectrum from this data as we know their begin and end time and what type of burst it is. We also have the exact time of bursts. But we will not use this data directly as out training set as the data might be missing other noises that we might find in the real life generated dynamic spectrum. So we would use these for GAN and generate synthetic data which is similar to what we have in real life. 

Alternatevily, I would also run this code on unlabelled dataset to find any overlaps. But for this we don not have timestamps to confirm the presence of a burst in the unlabelled dataset. So we might have to use another way to find them. Maybe we can use the patterns of burst we create now from synthetic dataset and try to match it in the unlabeeled dataseet. I am not sure if this would work :(

Import csv file for type 2 for labelling

In [ ]:
import csv

# Dictionary to store formatted dates and type 2 label
date_to_type = {}

with open('type 2 list found in catalogue.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Skip header row
    
    for row in csv_reader:
        if len(row) >= 2:
            # Extract date and time from second column (e.g., "20/04/22 12:00")
            date_time_str = row[1].strip()
            
            # Parse date (DD/MM/YY) and time (HH:MM)
            date_part, time_part = date_time_str.split(' ')
            day, month, year = date_part.split('/')
            
            # Convert to YYYY-MM-DD format (assuming 20XX for years)
            year = f"20{year}"
            
            # Format the key as per requirement: YYYY-MM-DD_HH:MM:00.000000
            formatted_key = f"{year}-{month}-{day}_{time_part}:00.000000"
            
            # Set type 2 for all entries
            date_to_type[formatted_key] = 2
    
# Print first 5 entries to verify
for i, (key, value) in enumerate(date_to_type.items()):
    if i >= 5:
        break
    print(f"'{key}': {value}")

print(f"\nTotal entries: {len(date_to_type)}")

'2022-04-20_12:00:00.000000': 2
'2022-04-20_12:15:00.000000': 2
'2022-04-22_13:15:00.000000': 2
'2022-04-22_13:30:00.000000': 2
'2022-05-25_18:15:00.000000': 2

Total entries: 128


Import csv for new dataset for comparision 

In [9]:
import csv

# Path to your CSV file
csv_file = 'dates_and_urls_of_new_data.csv'

date_dict = {}

with open(csv_file, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Get the Date column value
        full_date = row['Date']
        # Split by '_' and take the last two parts (date and time)
        parts = full_date.split('_')
        if len(parts) >= 3:
            date_part = parts[-2]  # e.g., '2022-05-12'
            time_part = parts[-1]  # e.g., '13:00:00.000000'
            # Remove any trailing whitespace or newlines
            date_part = date_part.strip()
            time_part = time_part.strip()
            # Format as 'YYYYMMDD_HH:MM:SS.ffffff'
            formatted_key = f"{date_part}_{time_part}"
            # Add to dictionary with empty string or None as value
            date_dict[formatted_key] = None  # or use None

# Print first 5 entries to verify
for i, (key, value) in enumerate(date_dict.items()):
    if i >= 5:
        break
    print(f"'{key}': {value}")

print(f"\nTotal entries: {len(date_dict)}")

'2022-05-12_13:00:00.000000': None
'2022-05-12_13:30:00.000000': None
'2022-05-12_12:00:00.000000': None
'2022-05-12_07:00:00.000000': None
'2022-05-12_07:30:00.000000': None

Total entries: 47667


Now since, we have type 2 list from cataloge, and labels from offline dataset and solarmonitor labels. We can find the overlaps in older and newer dataset and make a final dataset which will be divided in 2 parts

1. Labelled Dataset: which was made from all the data and labels from the overlap of data in older and newer data. Labels will be used from all the 3 sources as metioned above. 

2. Unlabelled dataset: All the remaining data for which we were not able to find labels for from both the datasets. 

### Make a labels file

First make a function to approximate minutes and seconds both similar to the function approximate_minutes. It should take the time in this 2022-05-02_06:49:00.00000 format and approximate the time part. We are doing this to homogenize the times which are different for labels and data images

In [11]:
def approximate_minutes_seconds(timestamp):
    """
    Approximate the time part of a timestamp string (YYYY-MM-DD_HH:MM:SS.ffffff)
    to the nearest 15-minute interval, setting seconds and microseconds to zero.
    Returns a string in the same format with approximated time.
    """
    from datetime import datetime, timedelta

    # Parse the timestamp
    try:
        dt = datetime.strptime(timestamp, "%Y-%m-%d_%H:%M:%S.%f")
    except ValueError:
        # If microseconds are missing, try without them
        dt = datetime.strptime(timestamp, "%Y-%m-%d_%H:%M:%S")

    # Calculate the number of minutes since the hour
    minute = dt.minute
    # Find the closest 15-min interval
    intervals = [0, 15, 30, 45]
    closest = min(intervals, key=lambda x: abs(x - minute))

    # If rounding up to 60, increment hour
    if minute >= 52.5:
        dt = dt.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)
    else:
        dt = dt.replace(minute=closest, second=0, microsecond=0)

    # Return in the same format
    return dt.strftime("%Y-%m-%d_%H:%M:%S.%f")[:-3] + "000"

# Example usage:
print(approximate_minutes_seconds("2022-05-02_06:49:00.000000"))  # 2022-05-02_06:45:00.000000
print(approximate_minutes_seconds("2022-05-02_06:53:12.123456"))  # 2022-05-02_07:00:00.000000

2022-05-02_06:45:00.000000
2022-05-02_07:00:00.000000
